In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

shared = [
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Fwd Packets Length Total",
    "Bwd Packets Length Total",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Avg Packet Size"
]

print("Shared features:", len(shared))

Shared features: 8


In [2]:
# Load cleaned CICIDS2017 dataset
cic = pd.read_parquet("../data/clean.parquet")

print("CICIDS2017 shape:", cic.shape)

# Verify shared features
missing = [c for c in shared if c not in cic.columns]

print("Missing features:", missing)

# Select the eight features
Xc = cic[shared]

# Convert multiclass labels into binary labels:
# 0 = BENIGN
# 1 = ATTACK
yc = (cic["Group"] != "BENIGN").astype(int)

print("\nBinary label distribution:")
print(yc.value_counts())

CICIDS2017 shape: (2198955, 70)
Missing features: []

Binary label distribution:
Group
0    1862525
1     336430
Name: count, dtype: int64


In [3]:
# Split CICIDS2017 into training and testing sets
Xtr, Xte, ytr, yte = train_test_split(
    Xc,
    yc,
    test_size=0.2,
    stratify=yc,
    random_state=42
)

print("Training shape:", Xtr.shape)
print("Testing shape:", Xte.shape)

print("\nTraining labels:")
print(ytr.value_counts())

print("\nTesting labels:")
print(yte.value_counts())

Training shape: (1759164, 8)
Testing shape: (439791, 8)

Training labels:
Group
0    1490020
1     269144
Name: count, dtype: int64

Testing labels:
Group
0    372505
1     67286
Name: count, dtype: int64


In [4]:
small = RandomForestClassifier(
    n_estimators=50,
    max_depth=20,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

small.fit(Xtr, ytr)

print("Eight-feature Random Forest training completed!")

Eight-feature Random Forest training completed!


In [5]:
print("== Same dataset (CICIDS2017 test split) ==")

pred_cic = small.predict(Xte)

print(classification_report(
    yte,
    pred_cic,
    digits=4
))

print("Confusion Matrix:")
print(confusion_matrix(yte, pred_cic))

== Same dataset (CICIDS2017 test split) ==
              precision    recall  f1-score   support

           0     0.9977    0.9971    0.9974    372505
           1     0.9842    0.9871    0.9856     67286

    accuracy                         0.9956    439791
   macro avg     0.9909    0.9921    0.9915    439791
weighted avg     0.9956    0.9956    0.9956    439791

Confusion Matrix:
[[371437   1068]
 [   869  66417]]


In [6]:
# Load UNSW-NB15 testing dataset
u = pd.read_csv("../data/unsw/UNSW_NB15_testing-set.csv")

print("UNSW testing shape:", u.shape)

print("\nAvailable columns:")
print(u.columns.tolist())

print("\nLabel distribution:")
print(u["label"].value_counts())

print("\nAttack categories:")
print(u["attack_cat"].value_counts())

UNSW testing shape: (82332, 45)

Available columns:
['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label']

Label distribution:
label
1    45332
0    37000
Name: count, dtype: int64

Attack categories:
attack_cat
Normal            37000
Generic           18871
Exploits          11132
Fuzzers            6062
DoS                4089
Reconnaissance     3496
Analysis            677
Backdoor            583
Shellcode           378
Worms                44
Name: count, dtype: int64


In [7]:
# Convert UNSW-NB15 features into CICIDS2017 shared features

# Avoid division by zero
dur = u["dur"].replace(0, np.nan)

pkts = (
    u["spkts"] + u["dpkts"]
).replace(0, np.nan)

# Construct the eight shared features
Xu = pd.DataFrame({
    "Flow Duration": u["dur"] * 1e6,

    "Total Fwd Packets": u["spkts"],

    "Total Backward Packets": u["dpkts"],

    "Fwd Packets Length Total": u["sbytes"],

    "Bwd Packets Length Total": u["dbytes"],

    "Flow Bytes/s": (
        u["sbytes"] + u["dbytes"]
    ) / dur,

    "Flow Packets/s": (
        u["spkts"] + u["dpkts"]
    ) / dur,

    "Avg Packet Size": (
        u["sbytes"] + u["dbytes"]
    ) / pkts
})

# Handle infinite and missing values
Xu = Xu.replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

# Ensure the exact feature order expected by the model
Xu = Xu[shared]

# UNSW labels: 0 = normal, 1 = attack
yu = u["label"]

print("UNSW feature shape:", Xu.shape)

print("\nFeature columns:")
print(Xu.columns.tolist())

print("\nMissing values:", Xu.isnull().sum().sum())

print("\nLabel distribution:")
print(yu.value_counts())

UNSW feature shape: (82332, 8)

Feature columns:
['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Flow Bytes/s', 'Flow Packets/s', 'Avg Packet Size']

Missing values: 0

Label distribution:
label
1    45332
0    37000
Name: count, dtype: int64


In [8]:
print("== Cross-dataset (train CIC -> test UNSW) ==")

pu = small.predict(Xu)

print("\nClassification Report:")
print(classification_report(yu, pu, digits=4))

print("\nConfusion Matrix:")
print(confusion_matrix(yu, pu))

print("\nPredictions by UNSW attack category:")
print(
    pd.crosstab(
        u["attack_cat"],
        pd.Series(pu, index=u.index, name="predicted"),
        normalize="index"
    ).round(3)
)

== Cross-dataset (train CIC -> test UNSW) ==

Classification Report:
              precision    recall  f1-score   support

           0     0.4494    1.0000    0.6201     37000
           1     0.0000    0.0000    0.0000     45332

    accuracy                         0.4494     82332
   macro avg     0.2247    0.5000    0.3101     82332
weighted avg     0.2020    0.4494    0.2787     82332


Confusion Matrix:
[[37000     0]
 [45332     0]]

Predictions by UNSW attack category:
predicted         0
attack_cat         
Analysis        1.0
Backdoor        1.0
DoS             1.0
Exploits        1.0
Fuzzers         1.0
Generic         1.0
Normal          1.0
Reconnaissance  1.0
Shellcode       1.0
Worms           1.0


c:\Users\huawe\Documents\sentinelAI\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\huawe\Documents\sentinelAI\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\huawe\Documents\sentinelAI\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}

In [9]:
# Compare CICIDS2017 training features with UNSW-NB15 features

print("CICIDS2017 training feature ranges:")
display(Xtr[shared].describe().round(2))

print("\nUNSW-NB15 feature ranges:")
display(Xu[shared].describe().round(2))

# Check how many UNSW values fall outside CIC training ranges
print("\nPercentage of UNSW values outside CIC training ranges:")

for col in shared:
    cic_min = Xtr[col].min()
    cic_max = Xtr[col].max()

    outside = (
        (Xu[col] < cic_min) |
        (Xu[col] > cic_max)
    ).mean() * 100

    print(f"{col}: {outside:.2f}%")

# Inspect model probabilities
probabilities = small.predict_proba(Xu)[:, 1]

print("\nAttack probability statistics:")
print(pd.Series(probabilities).describe().round(4))

print("\nHighest attack probability:", probabilities.max())

CICIDS2017 training feature ranges:


,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Flow Bytes/s,Flow Packets/s,Avg Packet Size
count,1.759164e+06,1759164.00,1759164.00,1759164.00,1.759164e+06,1.759164e+06,1759164.00,1759164.00
mean,1.900373e+07,11.54,13.11,693.27,2.076429e+04,3.901010e+05,15790.04,240.31
std,3.709426e+07,842.42,1118.85,11638.80,2.550898e+06,6.049470e+06,106199.73,359.00
min,-1.300000e+01,1.00,0.00,0.00,0.000000e+00,-2.610000e+08,-2000000.00,0.00
25%,6.963500e+03,2.00,1.00,41.00,4.600000e+01,1.190100e+02,1.18,39.47
50%,8.250450e+04,2.00,2.00,76.00,1.940000e+02,2.738900e+03,49.14,90.75
75%,6.288517e+06,7.00,6.00,368.00,3.232000e+03,1.801802e+04,567.87,227.65
max,1.200000e+08,219759.00,291922.00,12900000.00,6.554531e+08,2.071000e+09,4000000.00,3893.33



UNSW-NB15 feature ranges:


,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Flow Bytes/s,Flow Packets/s,Avg Packet Size
count,82332.00,82332.00,82332.00,82332.00,82332.00,8.233200e+04,82332.00,82332.00
mean,1006756.15,18.67,17.55,7993.91,13233.79,1.637046e+07,166171.40,169.31
std,4710444.41,133.92,115.57,171642.26,151471.46,4.510520e+07,300448.80,214.57
min,0.00,1.00,0.00,24.00,0.00,0.000000e+00,0.00,24.00
25%,8.00,2.00,0.00,114.00,0.00,3.270050e+03,29.93,57.00
50%,14138.00,6.00,2.00,534.00,178.00,3.053723e+05,3217.46,84.00
75%,719360.25,12.00,10.00,1280.00,956.00,1.628571e+07,250000.00,135.50
max,59999989.00,10646.00,11018.00,14355774.00,14657531.00,1.317000e+09,2000000.00,1504.00



Percentage of UNSW values outside CIC training ranges:
Flow Duration: 0.00%
Total Fwd Packets: 0.00%
Total Backward Packets: 0.00%
Fwd Packets Length Total: 0.00%
Bwd Packets Length Total: 0.00%
Flow Bytes/s: 0.00%
Flow Packets/s: 0.00%
Avg Packet Size: 0.00%

Attack probability statistics:
count    82332.0000
mean         0.0069
std          0.0272
min          0.0000
25%          0.0000
50%          0.0000
75%          0.0001
max          0.4800
dtype: float64

Highest attack probability: 0.48


In [10]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Evaluate the model's probability ranking
print("ROC-AUC:", round(roc_auc_score(yu, probabilities), 4))
print("Average Precision:", round(
    average_precision_score(yu, probabilities), 4
))

# Compare several classification thresholds
thresholds = [0.50, 0.40, 0.30, 0.20, 0.10, 0.05, 0.01]

for threshold in thresholds:
    predictions = (probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        yu, predictions, labels=[0, 1]
    ).ravel()

    print(f"\nThreshold: {threshold}")
    print("Predicted attacks:", int(predictions.sum()))
    print("Precision:", round(
        precision_score(yu, predictions, zero_division=0), 4
    ))
    print("Recall:", round(
        recall_score(yu, predictions, zero_division=0), 4
    ))
    print("F1:", round(
        f1_score(yu, predictions, zero_division=0), 4
    ))
    print("False-positive rate:", round(
        fp / (fp + tn), 4
    ))

ROC-AUC: 0.4007
Average Precision: 0.5148

Threshold: 0.5
Predicted attacks: 0
Precision: 0.0
Recall: 0.0
F1: 0.0
False-positive rate: 0.0

Threshold: 0.4
Predicted attacks: 2
Precision: 0.5
Recall: 0.0
F1: 0.0
False-positive rate: 0.0

Threshold: 0.3
Predicted attacks: 25
Precision: 0.96
Recall: 0.0005
F1: 0.0011
False-positive rate: 0.0

Threshold: 0.2
Predicted attacks: 990
Precision: 0.0929
Recall: 0.002
F1: 0.004
False-positive rate: 0.0243

Threshold: 0.1
Predicted attacks: 1729
Precision: 0.2389
Recall: 0.0091
F1: 0.0176
False-positive rate: 0.0356

Threshold: 0.05
Predicted attacks: 3032
Precision: 0.342
Recall: 0.0229
F1: 0.0429
False-positive rate: 0.0539

Threshold: 0.01
Predicted attacks: 9540
Precision: 0.3398
Recall: 0.0715
F1: 0.1182
False-positive rate: 0.1702


In [11]:
thresholds = [0.50, 0.40, 0.30, 0.20, 0.10, 0.05, 0.01]

results = []

for threshold in thresholds:
    predictions = (probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        yu, predictions, labels=[0, 1]
    ).ravel()

    results.append({
        "Threshold": threshold,
        "Predicted Attacks": int(predictions.sum()),
        "Precision": precision_score(
            yu, predictions, zero_division=0
        ),
        "Recall": recall_score(
            yu, predictions, zero_division=0
        ),
        "F1": f1_score(
            yu, predictions, zero_division=0
        ),
        "FPR": fp / (fp + tn)
    })

threshold_results = pd.DataFrame(results)

display(threshold_results.round(4))

,Threshold,Predicted Attacks,Precision,Recall,F1,FPR
0,0.50,0,0.0000,0.0000,0.0000,0.0000
1,0.40,2,0.5000,0.0000,0.0000,0.0000
2,0.30,25,0.9600,0.0005,0.0011,0.0000
3,0.20,990,0.0929,0.0020,0.0040,0.0243
4,0.10,1729,0.2389,0.0091,0.0176,0.0356
5,0.05,3032,0.3420,0.0229,0.0429,0.0539
6,0.01,9540,0.3398,0.0715,0.1182,0.1702
